## Import and Install

In [1]:
import os
import warnings
import pandas as pd
import torch
import numpy as np

!pip -qq install peft
!pip -qq install bitsandbytes
!pip -qq install GPUtil
!pip -qq install transformers==4.43.1
!pip -qq install transformers accelerate sentencepiece
!pip -qq install accelerate==0.34.2
!pip -qq install torch
!pip -qq install langdetect

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 26.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 23.8 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 70.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.2 MB/s eta

## Configure

In [2]:
from transformers import set_seed

SEED = 123
set_seed(SEED)

warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

#INPUT_DIR = '/kaggle/input/2025-jbnu-ai-competition'
INPUT_DIR = '/kaggle/input/2025-sw-digital-competition-datasets'

DIR = '/kaggle/working/'
NUM_WORKERS = os.cpu_count()
NUM_CLASSES = 2

EPOCHS, R, LORA_ALPHA, LORA_DROPOUT = 1, 64, 32, 0.3
BATCH_SIZE = 6

#MODEL_ID = '/kaggle/input/gemma-2/transformers/gemma-2-9b/2' # 7/2 Gemma 2b V2 모델 사용
MODEL_ID = '/kaggle/input/llama-2/pytorch/7b-hf/1'

2025-07-14 16:49:37.697498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752511778.054881      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752511778.150448      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Data processing

In [3]:
#from datasets import load_dataset, load
#import torch

In [4]:
from datasets import load_dataset, concatenate_datasets

# 1. 데이터 로드
dataset_53 = load_dataset('csv', data_files='/kaggle/input/digital-competition-no-train-dataset/non_train_human_token.csv')
dataset_ai = load_dataset('csv', data_files='/kaggle/input/digital-competition-no-train-dataset/top20000_ai_like_sentences (1).csv')

# 2. 라벨 기준 필터링
human_data = dataset_53['train'].filter(lambda x: x['label'] == 0)
ai_data = dataset_ai['train'].filter(lambda x: x['label'] == 1)

ai_data_filtered_by_generated = ai_data.filter(lambda x: x['generated'] >= 0.6)

human_data = human_data.remove_columns(['text_length'])
human_data = human_data.remove_columns(['token_count'])
human_data = human_data.remove_columns(['token_bin'])
ai_data_filtered_by_generated = ai_data_filtered_by_generated.remove_columns(['generated'])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/15650 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16733 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16733 [00:00<?, ? examples/s]

In [5]:
# 4. 합치고 섞기
combined_dataset = concatenate_datasets([human_data, ai_data_filtered_by_generated]).shuffle(seed=42)
test_dataset = combined_dataset.select(range(2000))

# 5. DatasetDict 형태로 재구성해서 train에 할당
from datasets import DatasetDict

dataset = DatasetDict({
    'train': combined_dataset,
    'test' : test_dataset
})

In [6]:
print(dataset, dataset.keys())
dataset["train"][0], dataset['test'][:5]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 24332
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
}) dict_keys(['train', 'test'])


({'text': '형제는 모두 네 명입니다. 저 위로 형이 두 명 아래로는 여동생이 한 명 있어요. 큰 형은 소우타 둘째 형은 요우타 그리고 셋째는 코우타 막내는 여동생 히나타입니다.',
  'label': 1},
 {'text': ['형제는 모두 네 명입니다. 저 위로 형이 두 명 아래로는 여동생이 한 명 있어요. 큰 형은 소우타 둘째 형은 요우타 그리고 셋째는 코우타 막내는 여동생 히나타입니다.',
   '2010년 3월 24일에 제1호 태풍 오마이스가 발생한 것을 시작으로 연말까지 총 14개의 태풍이 발생하였다. 평균적으로 1년 동안 태풍 발생 개수가 26.7개인 것과 비교하면 매우 낮은 수치이다.',
   '축복의 광장: 미니맵에서 9시에서 10시 방향에 위치한 건물로, 이 건물이 파괴되지 않으면 악마는 12시 왕좌 입성할 수 없다. 이 건물이 파괴되면 또한 신의 왕좌의 무적 상태가 해제된다.',
   '4월 13일, 도네츠크 인민 공화국의 시위대는 마리우폴의 행정 청사를 장악했다. 우크라이나 정부는 4월 24일 이 건물을 "해방시켰다"라고 발표했지만, 건물 주변의 BBC의 인터뷰는 지역 주민들이 막아버렸다.',
   '시즌 15승 4패, 2점대 평균자책점으로 세스 후랭코프와 함께 뛰어난 활약을 펼치면서 두산의 4시즌 연속 한국시리즈 진출을 이끌었고 이에 힘입어 외국인 선수로는 최초로 최동원상을 수상했다.'],
  'label': [1, 0, 0, 0, 0]})

## Tokenize Data

In [7]:
## **모델을 불러올때 다국어 모델을 불러와야 함**
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

tokenizer.pad_token = tokenizer.eos_token
print(tokenizer.padding_side, tokenizer.pad_token)

left </s>


In [8]:
tokenized_dataset = {}

MAX_LENGTH = 512

for split in dataset.keys():
    tokenized_dataset[split] = dataset[split].map(
        lambda x: tokenizer(x["text"], truncation=True, max_length=MAX_LENGTH), batched=True
    )

tokenized_dataset["train"], tokenized_dataset["test"]

Map:   0%|          | 0/24332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

(Dataset({
     features: ['text', 'label', 'input_ids', 'attention_mask'],
     num_rows: 24332
 }),
 Dataset({
     features: ['text', 'label', 'input_ids', 'attention_mask'],
     num_rows: 2000
 }))

In [9]:
print(set(dataset['train']['label']))

{0, 1}


## Load and setup Model (GPU Need)

In [10]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig

NUM_CLASSES = 2
quant_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    quantization_config=quant_config,
    device_map="auto"
)
print(model.config.pad_token_id)
model.config.pad_token_id = model.config.eos_token_id
print(model.config.pad_token_id)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/llama-2/pytorch/7b-hf/1 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


0
2


In [11]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

## LoRA config and LoRA model setting

In [12]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    task_type=TaskType.SEQ_CLS,
    target_modules='all-linear'
)
lora_config

LoraConfig(task_type=<TaskType.SEQ_CLS: 'SEQ_CLS'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=64, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.3, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, eva_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)

In [13]:
lora_model = get_peft_model(model, lora_config)
lora_model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): LlamaForSequenceClassification(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.3, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict(

In [14]:
lora_model.print_trainable_parameters()

trainable params: 159,916,032 || all params: 6,767,267,840 || trainable%: 2.3631


## Accelerate Setting (use multiple GPU)

In [15]:
from accelerate import Accelerator

accelerator = Accelerator()

Lora_model = accelerator.prepare(
    lora_model
)

## Trainer setting

In [16]:
import torch
import numpy as np
import torch.nn.functional as F
from transformers import Trainer
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}


class SoftLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")          # float soft label (0.0 ~ 1.0)
        outputs = model(**inputs)
        logits = outputs.logits                # shape: [B, 2]

        # soft label: sigmoid-style → BCE
        probs = torch.softmax(logits, dim=-1)  # shape: [B, 2]
        ai_probs = probs[:, 1]                 # "AI일 확률"

        loss = F.binary_cross_entropy(ai_probs, labels.float())
        return (loss, outputs) if return_outputs else loss


trainer = SoftLabelTrainer(
    model=Lora_model,    
    # 학습을 어떻게 진행할지 결정
    args=TrainingArguments(
        output_dir="./data/", # 모델 체크포인트 저장 위치
        learning_rate=2e-5,   # 학습률
        per_device_train_batch_size=BATCH_SIZE, # 학습 배치 크기 BATCH_SIZE
        per_device_eval_batch_size=BATCH_SIZE,  # 평가 배치 크기 BATCH_SIZE
        evaluation_strategy="epoch",  # 매 에폭마다 평가
        save_strategy="epoch", # 매 에폭마다 모델 저장
        num_train_epochs=EPOCHS, # 학습할 총 에폭 수 EPOCHS
        weight_decay=0.01, # 가중치 감소 (L2 정규화)
        load_best_model_at_end=True, #가장 성능 좋은 모델을 마지막에 로드
        logging_steps=10, # 로그 출력 빈도
        report_to="none", # 로그 기록 툴 사용 안함
        fp16=True
    ),
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

## Train and save model

In [17]:
print("Training the Model...")
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Training the Model...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.078200,0.183971,0.943500


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


TrainOutput(global_step=4056, training_loss=0.3406422878330834, metrics={'train_runtime': 20488.4244, 'train_samples_per_second': 1.188, 'train_steps_per_second': 0.198, 'total_flos': 1.9468440503672832e+17, 'train_loss': 0.3406422878330834, 'epoch': 1.0})

In [18]:
#print("Saving the Model...")
#lora_model.save_pretrained('fine-tuned-model')

## Makeing Predictions

In [19]:
from transformers import pipeline

clf = pipeline("text-classification", lora_model, tokenizer=MODEL_ID)

The model 'PeftModelForSequenceClassification' is not supported for text-classification. Supported models are ['AlbertForSequenceClassification', 'BartForSequenceClassification', 'BertForSequenceClassification', 'BigBirdForSequenceClassification', 'BigBirdPegasusForSequenceClassification', 'BioGptForSequenceClassification', 'BloomForSequenceClassification', 'CamembertForSequenceClassification', 'CanineForSequenceClassification', 'LlamaForSequenceClassification', 'ConvBertForSequenceClassification', 'CTRLForSequenceClassification', 'Data2VecTextForSequenceClassification', 'DebertaForSequenceClassification', 'DebertaV2ForSequenceClassification', 'DistilBertForSequenceClassification', 'ElectraForSequenceClassification', 'ErnieForSequenceClassification', 'ErnieMForSequenceClassification', 'EsmForSequenceClassification', 'FalconForSequenceClassification', 'FlaubertForSequenceClassification', 'FNetForSequenceClassification', 'FunnelForSequenceClassification', 'GemmaForSequenceClassification'

In [20]:
test_df = pd.read_csv(f"/kaggle/input/2025-sw-digital-competition-datasets/test.csv")
sample_submission = pd.read_csv("/kaggle/input/2025-sw-digital-competition-datasets/sample_submission.csv")

display(test_df.head())
display(sample_submission.head())

,ID,title,paragraph_index,paragraph_text
0,TEST_0000,공중 도덕의 의의와 필요성,0,도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다. 그러므로 도덕...
1,TEST_0001,공중 도덕의 의의와 필요성,1,도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없다. 개인적인 측면과 사회적...
2,TEST_0002,공중 도덕의 의의와 필요성,2,"여기에 이른바 공중도덕은 실천적, 사회적 도덕의 한 부문이다. 즉, 공중 도덕이라 ..."
3,TEST_0003,공중 도덕의 의의와 필요성,3,우리가 공동 생활을 하는 데 있어서 공중 도덕이 필요함은 위에서 말한 것처럼 알 수...
4,TEST_0004,풍습과 그 개선,0,인간 사회에서는 다 함께 지켜야 할 어떤 기준이 있어 이를 따르면 옳다고 하고 따르...


,ID,generated
0,TEST_0000,0
1,TEST_0001,0
2,TEST_0002,0
3,TEST_0003,0
4,TEST_0004,0


In [21]:
from tqdm import tqdm

predictions = []

print("Making prediction on test dataset...")

for text in tqdm(test_df['paragraph_text'].values):
    prediction = clf(text)
    score = prediction[0]['score']  # 확률값 (0~1)
    label = prediction[0]['label']
    
    # 만약 LABEL_1의 확률이 아니라면 뒤집어야 함
    if label == 'LABEL_0':
        score = 1 - score  # LABEL_1의 확률로 보정
    
    predictions.append(score)

sample_submission['generated'] = predictions

Making prediction on test dataset...


100%|██████████| 1962/1962 [36:19<00:00,  1.11s/it]


In [22]:
try:
    sample_submission = sample_submission.drop(['target'], axis=1)
    print(sample_submission)
    
except:
    print("No have target axis")
    print(sample_submission)

No have target axis
             ID  generated
0     TEST_0000   0.388850
1     TEST_0001   0.003562
2     TEST_0002   0.000742
3     TEST_0003   0.042463
4     TEST_0004   0.230486
...         ...        ...
1957  TEST_1957   0.599251
1958  TEST_1958   0.578686
1959  TEST_1959   0.000151
1960  TEST_1960   0.001348
1961  TEST_1961   0.000019

[1962 rows x 2 columns]


In [23]:
sample_submission.to_csv(f'submission.csv', index=False)

In [24]:
# 체크포인트 폴더 zip 파일로 저장

import os
import zipfile

target_folder = '/kaggle/working/data/checkpoint-4056' # 체크포인트에 맞는 숫자 기입
with zipfile.ZipFile(os.path.join(target_folder, 'compress.zip'), 'w') as f:
    f.write(os.path.join(target_folder, 'README.md'))
    f.write(os.path.join(target_folder, 'adapter_config.json'))
    f.write(os.path.join(target_folder, 'adapter_model.safetensors'))
    f.write(os.path.join(target_folder, 'optimizer.pt'))
    f.write(os.path.join(target_folder, 'rng_state.pth'))
    f.write(os.path.join(target_folder, 'scheduler.pt'))
    f.write(os.path.join(target_folder, 'special_tokens_map.json'))
    f.write(os.path.join(target_folder, 'tokenizer.json'))
    f.write(os.path.join(target_folder, 'tokenizer.model'))
    f.write(os.path.join(target_folder, 'tokenizer_config.json'))
    f.write(os.path.join(target_folder, 'trainer_state.json'))
    f.write(os.path.join(target_folder, 'training_args.bin'))
